# Titanic 생존 예측 튜토리얼

본 튜토리얼에서는 Titanic 데이터를 활용하여 머신러닝의 기본 흐름을 학습합니다.

**목표**
- 데이터 전처리 이해
- 모델 학습 과정 이해
- F1 Score 기반 성능 평가

## 전체 진행 과정

1. 데이터 불러오기  
2. 피처 / 타겟 분리  
3. Train / Test 분리  
4. 전처리  
5. 모델 학습  
6. 성능 평가  

> ⚠️ 이번 튜토리얼에서는 결과 재현을 위해 seed를 고정합니다.

In [80]:
import random
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

# seed 고정 (변경하지 마세요.)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 데이터 불러오기

In [82]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 피처 / 타겟 분리

- `survived` : 예측해야 할 정답(label)
- 나머지 컬럼 : 입력 변수(feature)

> ⚠️ `survived`를 입력 변수에 포함하면 안 됩니다.

In [84]:
X = df.drop(columns=["survived"])
y = df["survived"]

## 사용할 컬럼 선택

학습에 사용할 컬럼을 선택합니다.

> 💡 이 부분은 자유롭게 수정 가능합니다.  
> 다양한 컬럼을 사용해 성능을 개선해보세요!

In [86]:
numeric_features = ["age", "sibsp", "parch", "fare", "pclass"]
categorical_features = ["sex", "embarked", "alone"]

selected_features = numeric_features + categorical_features
X = X[selected_features]

## Train / Test 분리

- Train: 모델 학습용
- Test: 모델 성능 평가 및 예측 결과 생성

> ⚠️ 실제 공모전에서는 Test 데이터의 정답(label)을 확인할 수 없지만,  
> 이번 튜토리얼에서는 학습을 위해 Test label을 함께 사용하여 성능을 확인합니다.

In [88]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=SEED
)

## 전처리

모델이 데이터를 학습할 수 있도록 다음과 같은 처리를 수행합니다.

- 결측치 처리
- 범주형 변수 인코딩

> 💡 전처리 방식은 자유롭게 변경 가능합니다.

In [90]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## 모델 정의

RandomForest 모델을 사용합니다.

> 💡 모델은 자유롭게 변경 가능합니다.

In [92]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=SEED
)

clf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

## 모델 학습

In [94]:
clf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'sibsp', 'parch',
                                                   'fare', 'pclass']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['sex', 'embarked',
                                                   'alone'])])),
                ('model',
                 RandomForestClassifier(max_depth=5, n_estimators=200,
                                        random_state=42))])

## 성능 평가

이번 튜토리얼에서는 **F1 Score**를 사용합니다.

> 💡 Accuracy와 비교해보는 것도 추천합니다.

In [96]:
y_pred = clf.predict(X_test)

print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

F1 Score: 0.7150837988826816
              precision    recall  f1-score   support

           0       0.80      0.93      0.86       165
           1       0.84      0.62      0.72       103

    accuracy                           0.81       268
   macro avg       0.82      0.77      0.79       268
weighted avg       0.81      0.81      0.80       268



## 제출 파일 생성

모델이 예측한 결과를 바탕으로 제출 파일을 생성합니다.

제출 파일은 다음과 같은 형식을 따라야 합니다.

| PassengerId | Survived |
|-------------|----------|
| 1           | 0        |
| 2           | 1        |
| ...         | ...      |

- PassengerId: 각 데이터의 고유 ID (튜토리얼에서는 임의로 생성)
- Survived: 모델이 예측한 생존 여부 (0 또는 1)

> ⚠️ seaborn Titanic 데이터에는 PassengerId가 포함되어 있지 않기 때문에,  
> 제출 형식에 맞추기 위해 index를 활용하여 임의로 생성합니다.

In [98]:
submission = X_test.copy()

submission = submission.reset_index(drop=True)
submission["PassengerId"] = submission.index + 1
submission["Survived"] = y_pred

submission = submission[["PassengerId", "Survived"]]
submission.head()

,PassengerId,Survived
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


## 제출 파일 저장

생성한 제출 파일을 CSV 형식으로 저장합니다.

저장된 `submission.csv` 파일을 제출하면 됩니다.

> 💡 실제 공모전에서는 test 데이터에 대해 예측한 결과를 제출하게 됩니다.

In [100]:
submission.to_csv("submission.csv", index=False)